In [ ]:
#Étape 1 :impoter data set:

In [ ]:
import os

if os.path.exists("final_training_data.jsonl"):
    os.rename("final_training_data.jsonl", "train.jsonl")
    print("importation faite avec succées!")

importation faite avec succées!


In [ ]:
#Étape 2 : Préparation de l'environnement (Fix SQLite)

In [ ]:
!pip install -U autotrain-advanced pysqlite3-binary

import sys
__import__('pysqlite3')
sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')
print("✅ Fix SQLite appliqué !")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 80.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.2/78.2 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.6/51.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 341.3/341.3 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.4/336.4 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.9/269.9 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.5/225.5 kB 15.4 MB/s eta 

✅ Fix SQLite appliqué !


In [ ]:
#Étape 3 : Formatage ChatML (Spécifique à SmolLM2)

In [ ]:
from transformers import AutoTokenizer
import json

# 1. Charger le tokenizer
model_id = "HuggingFaceTB/SmolLM2-360M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 2. Transformer le JSONL vers le format 'text' attendu par AutoTrain
formatted_data = []
with open("train.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        # On applique le template de chat du modèle si nécessaire
        if "messages" in data:
            full_text = tokenizer.apply_chat_template(data["messages"], tokenize=False)
        elif "text" in data:
            full_text = data["text"]
        else:
            print(f"Warning: Skipping line with unexpected format: {line.strip()}")
            continue
        formatted_data.append({"text": full_text})

# 3. Écraser le fichier train.jsonl avec le nouveau format
with open("train.jsonl", "w", encoding="utf-8") as f:
    for entry in formatted_data:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"✅ Correction terminée ! Fichier prêt avec {len(formatted_data)} lignes au format ChatML.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

✅ Correction terminée ! Fichier prêt avec 51357 lignes au format ChatML.


In [ ]:
#Étape 4 : Lancer le Fine-Tuning

In [ ]:
# On supprime les anciens dossiers pour éviter les erreurs
!rm -rf professional-assistant

!autotrain llm --train \
    --model "HuggingFaceTB/SmolLM2-360M-Instruct" \
    --data-path . \
    --train-split "train" \
    --text-column "text" \
    --lr 1e-4 \
    --batch-size 4 \
    --epochs 1 \
    --trainer sft \
    --model-max-length 512 \
    --project-name "professional-assistant" \
    --mixed-precision "fp16" \
    --block-size 512 \
    --disable-gradient-checkpointing

INFO     | 2026-07-01 17:14:08 | autotrain.cli.run_llm:run:136 - Running LLM
WARNING  | 2026-07-01 17:14:08 | autotrain.trainers.common:__init__:286 - Parameters supplied but not used: config, backend, inference, func, deploy, version, train
Saving the dataset (1/1 shards): 100% 51357/51357 [00:00<00:00, 1142766.27 examples/s]
Saving the dataset (1/1 shards): 100% 51357/51357 [00:00<00:00, 1299079.52 examples/s]
INFO     | 2026-07-01 17:14:10 | autotrain.backends.local:create:20 - Starting local training...
WARNING  | 2026-07-01 17:14:10 | autotrain.commands:get_accelerate_command:58 - No GPU found. Forcing training on CPU. This will be super slow!
INFO     | 2026-07-01 17:14:10 | autotrain.commands:launch_command:514 - ['accelerate', 'launch', '--cpu', '-m', 'autotrain.trainers.clm', '--training_config', 'professional-assistant/training_params.json']
INFO     | 2026-07-01 17:14:10 | autotrain.commands:launch_command:515 - {'model': 'HuggingFaceTB/SmolLM2-360M-Instruct', 'project_name'

In [ ]:
#l'étape finale pour convertir votre modèle entraîné au format GGUF.

In [ ]:
# Installation de llama.cpp

In [ ]:
# 1. Cloner le dépôt officiel
!git clone https://github.com/ggerganov/llama.cpp

# 2. Installer les bibliothèques nécessaires
!pip install -r llama.cpp/requirements.txt
!pip install huggingface_hub

Cloning into 'llama.cpp'...
remote: Enumerating objects: 101547, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 101547 (delta 23), reused 6 (delta 5), pack-reused 101489 (from 3)
Receiving objects: 100% (101547/101547), 403.73 MiB | 22.87 MiB/s, done.
Resolving deltas: 100% (71452/71452), done.
Updating files: 100% (3018/3018), done.
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/

In [ ]:
import os
import shutil
from huggingface_hub import snapshot_download

# Configuration
target_dir = "professional-assistant" # Le nom de votre projet AutoTrain
base_model_id = "HuggingFaceTB/SmolLM2-360M-Instruct"
temp_repair = "repair_files"

print("⏳ Téléchargement des fichiers de structure manquants...")
# On télécharge uniquement la config et le tokenizer originaux
snapshot_download(
    repo_id=base_model_id,
    allow_patterns=["*.json", "*.model", "*.txt"],
    local_dir=temp_repair
)

print(f"🔧 Injection des fichiers dans {target_dir}...")
for file in os.listdir(temp_repair):
    src_path = os.path.join(temp_repair, file)
    dst_path = os.path.join(target_dir, file)

    # On ne copie que si le fichier n'existe pas déjà ou pour écraser les configs
    if os.path.isfile(src_path):
        shutil.copy(src_path, dst_path)
        print(f" ✅ Ajouté : {file}")

print("\n✨ Dossier prêt pour la conversion !")

⏳ Téléchargement des fichiers de structure manquants...


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

all_results.json:   0%|          | 0.00/786 [00:00<?, ?B/s]

eval_results.json:   0%|          | 0.00/589 [00:00<?, ?B/s]

trainer_state.json: 0.00B [00:00, ?B/s]

train_results.json:   0%|          | 0.00/232 [00:00<?, ?B/s]

🔧 Injection des fichiers dans professional-assistant...
 ✅ Ajouté : eval_results.json
 ✅ Ajouté : all_results.json
 ✅ Ajouté : tokenizer.json
 ✅ Ajouté : merges.txt
 ✅ Ajouté : train_results.json
 ✅ Ajouté : tokenizer_config.json
 ✅ Ajouté : vocab.json
 ✅ Ajouté : config.json
 ✅ Ajouté : special_tokens_map.json
 ✅ Ajouté : generation_config.json
 ✅ Ajouté : trainer_state.json

✨ Dossier prêt pour la conversion !


In [ ]:
# 3 : Conversion finale en GGUF

In [ ]:
# Lancement de la conversion
# --outtype f16 : pour garder toute la précision du modèle
!python3 llama.cpp/convert_hf_to_gguf.py {target_dir}/ \
  --outfile assistant_generique_final.gguf \
  --outtype f16

print("\n🚀 SUCCÈS ! Votre modèle est prêt : 'assistant_generique_final.gguf'")

INFO:hf-to-gguf:Loading model: professional-assistant
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:Set meta model
INFO:hf-to-gguf:Set model parameters
INFO:hf-to-gguf:gguf: context length = 8192
INFO:hf-to-gguf:gguf: embedding length = 960
INFO:hf-to-gguf:gguf: feed forward length = 2560
INFO:hf-to-gguf:gguf: head count = 15
INFO:hf-to-gguf:gguf: key-value head count = 5
INFO:hf-to-gguf:gguf: rope theta = 100000
INFO:hf-to-gguf:gguf: rms norm epsilon = 1e-05
INFO:hf-to-gguf:gguf: file type = 1
INFO:hf-to-gguf:Set model quantization version
INFO:hf-to-gguf:Set model tokenizer
INFO:numexpr.utils:NumExpr defaulting to 2 threads.
INFO:gguf.vocab:Adding 48900 merge(s).
INFO:gguf.vocab:Setting special token type bos to 1
INFO:gguf.vocab:Setting special token type eos to 2
INFO:gguf.vocab:Setting special token type unk to 0
INFO:gguf.vocab:Setting special token type 

In [ ]:
# 4 : Télécharger le modèle

In [ ]:
from google.colab import files
files.download('assistant_generique_final.gguf')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>